##### **Name: Arsh Ali**

##### **Course: 03-701**

##### **Assignment: Final Project: Bacterial Genome Analysis (Staphylococcus Condimenti)**

##### **Date: 10/05/2026**

#### Part A. Scan both genomic DNA strands for potential open reading frames longer than 50 codons

In [1]:
#import all packages/modules

from Bio.Seq import Seq #import seq
from Bio.SeqUtils import molecular_weight
from Bio import SeqIO #import SeqIO
from Bio.Data import CodonTable #import codon table
from Bio.Blast import NCBIWWW #import NCBIWWW
from Bio.Blast import NCBIXML #import NCBIXML
import re #import regular expression
import pandas as pd #import pandas  

table = CodonTable.unambiguous_dna_by_id[11] #print codon table
print(table)

Table 11 Bacterial, Archaeal, Plant Plastid

  |  T      |  C      |  A      |  G      |
--+---------+---------+---------+---------+--
T | TTT F   | TCT S   | TAT Y   | TGT C   | T
T | TTC F   | TCC S   | TAC Y   | TGC C   | C
T | TTA L   | TCA S   | TAA Stop| TGA Stop| A
T | TTG L(s)| TCG S   | TAG Stop| TGG W   | G
--+---------+---------+---------+---------+--
C | CTT L   | CCT P   | CAT H   | CGT R   | T
C | CTC L   | CCC P   | CAC H   | CGC R   | C
C | CTA L   | CCA P   | CAA Q   | CGA R   | A
C | CTG L(s)| CCG P   | CAG Q   | CGG R   | G
--+---------+---------+---------+---------+--
A | ATT I(s)| ACT T   | AAT N   | AGT S   | T
A | ATC I(s)| ACC T   | AAC N   | AGC S   | C
A | ATA I(s)| ACA T   | AAA K   | AGA R   | A
A | ATG M(s)| ACG T   | AAG K   | AGG R   | G
--+---------+---------+---------+---------+--
G | GTT V   | GCT A   | GAT D   | GGT G   | T
G | GTC V   | GCC A   | GAC D   | GGC G   | C
G | GTA V   | GCA A   | GAA E   | GGA G   | A
G | GTG V(s)| GCG A   | GAG E   | GGG

In [2]:
with open("/Users/arshali/CSB/python/GCF_900637235.1.fa") as f: #open FASTA file
    genome = list(SeqIO.parse(f, "fasta")) ##view each entry from the FASTA file and store DNA information/record
for record in genome: #iterate through each record in genome
    print(record.id, len(record)) #print each sequence ID and corresponding nucleotide length
print("Total genome records:", len(genome)) #check # of loaded entries/loaded genome records

NZ_LR134360.1 2650496
Total genome records: 1


In [3]:
#print the start and stop codons from table

start_codons = table.start_codons
stop_codons = table.stop_codons
print("Start Codons:", start_codons)
print("Stop Codons:", stop_codons)

Start Codons: ['TTG', 'CTG', 'ATT', 'ATC', 'ATA', 'ATG', 'GTG']
Stop Codons: ['TAA', 'TAG', 'TGA']


In [4]:
#First step: inspect forward and reverse-complement DNA sequences

dna = genome[0].seq.upper() #view chromosome DNA and change all DNA bases to uppercase
print(dna[:70]) #print first 70 nucleotides (forward sequence)
reverse_dna = dna.reverse_complement() #retrieve reverse complement of DNA 
print(reverse_dna[:70]) #print first 70 nucleotides of reverse complement

ATGTCTAAAGAAGAAATTTGGGACAAAGTTCTGGAATTAGCAAAGAATGAATTGACCCCTATTAGCTATA
ATAAACTCCATCTTTCTATAAAATTTTTTAGACTTATTCACAAAATCACACCCAGTGGATAATCAAGTAC


**Included in the Report:** The complete genome assembly of Staphylococcus Condimenti. The code below searches for possible protein-coding regions within the genome by checking/scanning three reading frames for each forward and reverse-complement strand (i.e scanning all six reading frames of a DNA sequence for start and stop codons). Each start codon has an in-frame stop codon and the coding length is calculated by counting the number of codons from the start codon up to its first in-frame stop codon (but excluding the stop given it signals the end of translation). If candidates had longer than 50 coding codons, they were saved as dictionaries containing the contig ID, strand, frame, position, length, DNA sequence, etc. DNA positions are zero-based (the first base is position zero) with an exclusive end, measured along either the forward or reverse-complement strand. 

Each start codon is paired with its first in-frame stop codon. The total potential ORFs identified in this code were 121,496, but these candidates will be further assessed by translating the genome's predicated proteins, calculating the molecular mass, and examining proteins shorter than 1,000 amino acids using BLAST. 

In [5]:
#Second step: Read/scan forward and reverse DNA strands to detect candidate ORFs

open_reading_frames = [] #store open reading frames
strands = { "+": dna, "-": reverse_dna} #create a dictionary storing forward and reverse-complement DNA 
for strand in strands: #iterate through each strand (forward and reverse-complement)
    strand_dna = strands[strand] #extract the stored DNA sequence for the + or - strand
    for frame in range(3): #iterate/check through each frame offset
    #group DNA at 3 different positions on each strand(0:first base, 1:second base, 3:third base)
        start_positions = [] #store possible start positions for this frame
        for i in range(frame, len(strand_dna) - 2, 3): #iterate through current frame in steps of 3 base-codons
            codon = strand_dna[i:i + 3] #retrieve current 3 base-codon
            if codon in start_codons: #if the codon is a start codon
                start_positions.append(i) #append its position (you can have multiple starts recorded)
            elif codon in stop_codons: #if the codon is instead a stop codon
                for start in start_positions:  #examine each recorded start before the stop codon 
                    codon_length = (i - start) // 3 #count bases from start but before start of stop codon
                    if codon_length > 50: #retrieve only possible ORFs longer than 50 coding codons
                        open_reading_frames.append({ #store and append all respective information about ORF of DNA in dictionary 
                            "Sequence ID": genome[0].id, #prints sequence ID
                            "Strand": strand, #print strand (+ or -)
                            "Frame": frame, #prints frame offset (codons grouped from index 0)
                            "Start": start, #prints start position (where the start codon of that ORF begins)
                            "Stop": i, #prints stop (where the stop codon of the ORF begins)
                            "End": i + 3, #prints end (immediately after stop codon)
                            "Length of ORF": codon_length, #prints coding codons (number of coding codons, so it includes the start but not the stop)
                            #number of codons encoding amino acids (includes start codon/excludes stop codon)
                            "DNA": strand_dna[start:i + 3]
                        })
                start_positions = [] #restart/clear list to ensure all starts are paired only with its first in-frame stop and not with a later stop codon
print("Potential ORFs:", len(open_reading_frames)) #print the number of open reading frames
orf_table = pd.DataFrame(open_reading_frames).astype({"DNA": str}) #show respective information about sequence in DataFrame (OpenAI,2026)
orf_table.head() #show first few lines of data 

Potential ORFs: 121496


,Sequence ID,Strand,Frame,Start,Stop,End,Length of ORF,DNA
0,NZ_LR134360.1,+,0,0,1359,1362,453,ATGTCTAAAGAAGAAATTTGGGACAAAGTTCTGGAATTAGCAAAGA...
1,NZ_LR134360.1,+,0,15,1359,1362,448,ATTTGGGACAAAGTTCTGGAATTAGCAAAGAATGAATTGACCCCTA...
2,NZ_LR134360.1,+,0,30,1359,1362,443,CTGGAATTAGCAAAGAATGAATTGACCCCTATTAGCTATATCACTT...
3,NZ_LR134360.1,+,0,51,1359,1362,436,TTGACCCCTATTAGCTATATCACTTGGTTTGAGCCGCCAAAAACGG...
4,NZ_LR134360.1,+,0,60,1359,1362,433,ATTAGCTATATCACTTGGTTTGAGCCGCCAAAAACGGAATTAATCG...


In [6]:
orf_table[orf_table["Strand"] == "-"].head()  #show data for reverse-complement strand

,Sequence ID,Strand,Frame,Start,Stop,End,Length of ORF,DNA
62804,NZ_LR134360.1,-,0,921,1275,1278,118,GTGATGGAAGTGGAAAAAGCATACCGTATTAAGAAGAACAGCGATT...
62805,NZ_LR134360.1,-,0,924,1275,1278,117,ATGGAAGTGGAAAAAGCATACCGTATTAAGAAGAACAGCGATTTCC...
62806,NZ_LR134360.1,-,0,930,1275,1278,115,GTGGAAAAAGCATACCGTATTAAGAAGAACAGCGATTTCCAAGCGA...
62807,NZ_LR134360.1,-,0,948,1275,1278,109,ATTAAGAAGAACAGCGATTTCCAAGCGATATATAGAAGAGGAAAGT...
62808,NZ_LR134360.1,-,0,975,1275,1278,100,ATATATAGAAGAGGAAAGTCGGTAGCAAATCGCCAGTTTGTTATTT...


In [7]:
orf_table["Strand"].value_counts() #show/validate number of ORFs were found for each strand

Strand
+    62804
-    58692
Name: count, dtype: int64

#### Part B. Translate the potential open reading frames into protein sequences

In [8]:
#Third step: Translate candidate ORFs into protein sequence

for orf in open_reading_frames: #iterate through each ORF dictionary in the list
    protein_sequence = orf["DNA"].translate(table=11, cds=True) #translate the DNA sequences of ORF into protein sequences using bacterial table 11
    #use cds to validate start codon, stop codon, and remove stop codon from result (OpenAI,2026)
    orf["Protein"] = protein_sequence #store protein sequence in orf dictionary 
    orf["Protein Sequence Length"] = len(protein_sequence) #count amino acids in protein sequence and store its length in dictionary
orf_table = pd.DataFrame(open_reading_frames).astype({"DNA": str, "Protein": str}) #add respective information about protein sequence in DataFrame
orf_table.head() #show first few lines of data

,Sequence ID,Strand,Frame,Start,Stop,End,Length of ORF,DNA,Protein,Protein Sequence Length
0,NZ_LR134360.1,+,0,0,1359,1362,453,ATGTCTAAAGAAGAAATTTGGGACAAAGTTCTGGAATTAGCAAAGA...,MSKEEIWDKVLELAKNELTPISYITWFEPPKTELIDIQENTAIILV...,453
1,NZ_LR134360.1,+,0,15,1359,1362,448,ATTTGGGACAAAGTTCTGGAATTAGCAAAGAATGAATTGACCCCTA...,MWDKVLELAKNELTPISYITWFEPPKTELIDIQENTAIILVESNFV...,448
2,NZ_LR134360.1,+,0,30,1359,1362,443,CTGGAATTAGCAAAGAATGAATTGACCCCTATTAGCTATATCACTT...,MELAKNELTPISYITWFEPPKTELIDIQENTAIILVESNFVQDFLK...,443
3,NZ_LR134360.1,+,0,51,1359,1362,436,TTGACCCCTATTAGCTATATCACTTGGTTTGAGCCGCCAAAAACGG...,MTPISYITWFEPPKTELIDIQENTAIILVESNFVQDFLKKQYTDII...,436
4,NZ_LR134360.1,+,0,60,1359,1362,433,ATTAGCTATATCACTTGGTTTGAGCCGCCAAAAACGGAATTAATCG...,MSYITWFEPPKTELIDIQENTAIILVESNFVQDFLKKQYTDIIADL...,433


#### Part C. Calculate the predicted molecular mass of each protein (in kD).

In [9]:
#Fourth step: Use imported BioPython Module to calculate molecular mass of each protein in KD

for orf in open_reading_frames:
    molecular_mass_da = molecular_weight(orf["Protein"], seq_type = "protein") #calculate predicted mass of each protein (OpenAI,2026)
    #store the result in Da and ord dictionary #seq type is protein to interpret the letters as the protein's amino acid sequence
    orf["Molecular Mass of Protein (kDa)"] = molecular_mass_da/1000 #convert the mass from Da to kD
orf_table = pd.DataFrame(open_reading_frames).astype({"DNA": str, "Protein": str}) #add respective information about protein sequence in DataFrame
orf_table.head() #show first few lines of data

,Sequence ID,Strand,Frame,Start,Stop,End,Length of ORF,DNA,Protein,Protein Sequence Length,Molecular Mass of Protein (kDa)
0,NZ_LR134360.1,+,0,0,1359,1362,453,ATGTCTAAAGAAGAAATTTGGGACAAAGTTCTGGAATTAGCAAAGA...,MSKEEIWDKVLELAKNELTPISYITWFEPPKTELIDIQENTAIILV...,453,52.181450
1,NZ_LR134360.1,+,0,15,1359,1362,448,ATTTGGGACAAAGTTCTGGAATTAGCAAAGAATGAATTGACCCCTA...,MWDKVLELAKNELTPISYITWFEPPKTELIDIQENTAIILVESNFV...,448,51.594815
2,NZ_LR134360.1,+,0,30,1359,1362,443,CTGGAATTAGCAAAGAATGAATTGACCCCTATTAGCTATATCACTT...,MELAKNELTPISYITWFEPPKTELIDIQENTAIILVESNFVQDFLK...,443,50.953057
3,NZ_LR134360.1,+,0,51,1359,1362,436,TTGACCCCTATTAGCTATATCACTTGGTTTGAGCCGCCAAAAACGG...,MTPISYITWFEPPKTELIDIQENTAIILVESNFVQDFLKKQYTDII...,436,50.155161
4,NZ_LR134360.1,+,0,60,1359,1362,433,ATTAGCTATATCACTTGGTTTGAGCCGCCAAAAACGGAATTAATCG...,MSYITWFEPPKTELIDIQENTAIILVESNFVQDFLKKQYTDIIADL...,433,49.843784


#### Part D. Blast 5 protein coding sequences at NCBI and return most similar hits. **** Choose sequences < 1000 amino acids long for this, otherwise, blast may not finish.

In [11]:
blast_candidates = (
     orf_table[(orf_table["Protein Sequence Length"] < 600) & (orf_table["Protein Sequence Length"] >= 200)] #choose protein sequences with < 1000 amino acids
     .drop_duplicates(subset = ["Sequence ID", "Strand", "Frame", "Stop"]) #choose different subsets of candidates (OpenAI,2026)
     .head(5) #show first 5 candidates 
)   #many candidiates will share the same stop; select candidates with new stops for BLAST analysis

blast_candidates #display candidates

,Sequence ID,Strand,Frame,Start,Stop,End,Length of ORF,DNA,Protein,Protein Sequence Length,Molecular Mass of Protein (kDa)
0,NZ_LR134360.1,+,0,0,1359,1362,453,ATGTCTAAAGAAGAAATTTGGGACAAAGTTCTGGAATTAGCAAAGA...,MSKEEIWDKVLELAKNELTPISYITWFEPPKTELIDIQENTAIILV...,453,52.181450
67,NZ_LR134360.1,+,0,13017,14322,14325,435,ATTGGAAAGGATGACCGTATTATGTTAGATATCAAATTATTCAGAA...,MGKDDRIMLDIKLFRNEPDYLKGKVKLRGMDPKVVDEVLELDGKRR...,435,49.517462
119,NZ_LR134360.1,+,0,16491,18276,18279,595,TTGGATAATTATGTAGATGATTTAAGCGGACAGATATCATCAGGCA...,MDNYVDDLSGQISSGNSIAIKSLPIGVIVLDENEDVEWMNHFMSDR...,595,67.271916
208,NZ_LR134360.1,+,0,18738,20151,20154,471,ATCAGAGGTGTAATCATAATGGATGGAATGTACGAACAAAACCAAA...,MRGVIIMDGMYEQNQMPHNNEAEQSVLGAIIIDPQLIGETQGVLLP...,471,52.963881
275,NZ_LR134360.1,+,0,24231,26028,26031,599,ATTGTTTATGTCTTGTTGATTATTATCGGTATGCAGATTATCGGTT...,MVYVLLIIIGMQIIGLYFTNNLERELTQTFKNNISQYAKQIEINIE...,599,68.396659


In [ ]:
from Bio.Blast import NCBIWWW #import NCBIWWW
from Bio.Blast import NCBIXML #import NCBIXML

selected_proteins = list(blast_candidates["Protein"]) #extract protein for DataFrame and store in list
blast_records = [] #create empty list to store BLAST records
for i in range(len(selected_proteins)): #iterate through all 5 proteins
    print("Protein:", i + 1) #print current protein number
    result_handle = NCBIWWW.qblast("blastp", "nr", selected_proteins[i], filter="L") #create handle for blast result
    #use low-complexity filtering to avoid repetitive segments
    filename = "protein" + str(i + 1) + "_blast.xml" #create filename for each protein result
    #save the result to a .xml file
    save_file = open(filename, "w") #open file in write node
    save_file.write(result_handle.read()) #write the xml text output into the file
    save_file.close() #close file after saved results
    result_handle.close() #close BLAST handle
    result_file = open(filename) #open xml file (read mode)
    blast_record = NCBIXML.read(result_file) #parse results BLAST search from file (alignments) into BLAST object
    result_file.close() #close file after parsing results
    blast_records.append(blast_record) #add current BLAST record to list 
    print("Number of alignments:", len(blast_record.alignments)) #print the # of alignments/matching protein sequences found each protein

* Note: The BLAST code ran sucessfully and all outputs/files were already saved for each protein as shown below. The cell may appear as unexecuted in the current session because I did not rerun the BLAST at the end due to the long runtime. However, it ran successfully, and the final results were saved and are shown in the last cell. 

In [26]:
for i in range(5):
    filename = "protein" + str(i + 1) + "_blast.xml"
    print(filename)

protein1_blast.xml
protein2_blast.xml
protein3_blast.xml
protein4_blast.xml
protein5_blast.xml


In [72]:
for i in range(len(blast_records)): #iterate through each blast record
    print("\nProtein:", i + 1) #print the current protein number
    print("Number of alignments:", len(blast_records[i].alignments)) #print the # of alignments/matching protein sequences found each protein
    for alignment in blast_records[i].alignments[:1]: #iterate through a given aligned protein sequence
        for hsp in alignment.hsps[:1]: #iterate through first aligned region for each protein hit
            print("****Top Alignment****") #print heading of top alignment
            print("sequence:", alignment.title) #print alignment info (protein information) 
            print("length:", alignment.length) #print length of BLAST protein 
            print("E value:", hsp.expect) #print the alignment score to see how common the sequences are
            print(hsp.query[:75] + "...") #print the sequence of protein from current analysis (the first 75)
            print(hsp.match[:75] + "...") #print matching amino acids
            print(hsp.sbjct[:75] + "...") #print the sequence of aligned protein from BLAST-NCBI database (the first 75)


Protein: 1
Number of alignments: 50
****Top Alignment****
sequence: ref|WP_047131985.1| chromosomal replication initiator protein DnaA [Staphylococcus condimenti] >gb|AMY05835.1| chromosomal replication initiation protein DnaA [Staphylococcus condimenti] >gb|PNZ63761.1| chromosomal replication initiator protein DnaA [Staphylococcus condimenti] >gb|QQS82363.1| chromosomal replication initiator protein DnaA [Staphylococcus condimenti] >gb|QRP95205.1| chromosomal replication initiator protein DnaA [Staphylococcus condimenti] >emb|VEG62970.1| chromosomal replication initiation protein [Staphylococcus condimenti]
length: 453
E value: 0.0
MSKEEIWDKVLELAKNELTPISYITWFEPPKTELIDIQENTAIILVESNFVQDFLKKQYTDIIADLIEKAIGTKL...
MSKEEIWDKVLELAKNELTPISYITWFEPPKTELIDIQENTAIILVESNFVQDFLKKQYTDIIADLIEKAIGTKL...
MSKEEIWDKVLELAKNELTPISYITWFEPPKTELIDIQENTAIILVESNFVQDFLKKQYTDIIADLIEKAIGTKL...

Protein: 2
Number of alignments: 50
****Top Alignment****
sequence: ref|WP_047131993.1| MULTISPECIES: serine--tRNA ligas

#### Citations

OpenAI. (2026). ChatGPT [Large language model]. https://chatgpt.com/

Usage: AI used as a tool to help determine which function (.astype) to use for changing DNA into a str  
Code: orf_table = pd.DataFrame(open_reading_frames).astype({"DNA": str})

Usage: AI used as a tool to understand the difference between cds and to.stop = TRUE in code    
Code: protein_sequence = orf["DNA"].translate(table=11, cds=True)

Usage: AI used as a tool to know if BioPython had Bio.SeqUtils for molecular_weight

Usage: AI used as a tool to understand how I could extract a combination of candidates with different stops for BLAST  
Code: .drop_duplicates(subset = ["Sequence ID", "Strand", "Frame", "Stop"])